# Build the classified ticket export for the dashboard

`outputs/tickets.csv` only carries ground-truth categories. The dashboard
needs to show what the pipeline actually *predicts* — a triage-time
category for every ticket, and a final-time category once a ticket closes
— plus where the two disagree, which is a useful signal on its own.

This uses the recommended `HybridRouter` from Phase 2 (TF-IDF + Logistic
Regression for known categories, embedding-centroid fallback for anything
it's never seen), trained on the full labeled dataset since this is a demo
export rather than a held-out evaluation — `02_classifier_comparison.ipynb`
remains the source of truth for accuracy numbers.

Output: `outputs/tickets_classified.csv`.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from taxonomy import load_taxonomy
from data_generation import generate_dataset
from classification_pipeline import build_classified_export

pd.set_option("display.max_colwidth", 80)

In [2]:
taxonomy = load_taxonomy()
tickets_df, conversations_df = generate_dataset(n_tickets=1200, taxonomy=taxonomy, seed=42)
print(f"{len(tickets_df)} tickets")

1200 tickets


In [3]:
classified_df = build_classified_export(tickets_df, conversations_df, taxonomy)
classified_df.head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,id,subject,status,priority,type,via_channel,group_id,created_at,updated_at,predicted_category_triage,predicted_category_final,predicted_group_triage,predicted_group_final,category_disagreement,taxonomy_version
0,10000,Package hasn't arrived yet,solved,low,task,web_widget,1,2025-06-26T00:48:15,2025-06-26T14:45:15,ORD-002,ORD-002,Order & shipping,Order & shipping,False,1
1,10001,Package hasn't arrived yet,solved,normal,incident,web_widget,1,2025-07-12T19:37:34,2025-07-13T01:53:34,ORD-002,ORD-002,Order & shipping,Order & shipping,False,1
2,10002,When will this be back in stock?,solved,normal,incident,web_widget,1,2025-08-24T05:32:40,2025-08-24T22:49:40,PRD-004,PRD-004,Product,Product,False,1
3,10003,Package hasn't arrived yet,solved,normal,question,web_widget,2,2025-06-14T21:53:40,2025-06-15T16:24:40,ORD-002,GEN-003,Order & shipping,General,True,1
4,10004,Address correction for #160738,solved,low,incident,web_widget,2,2025-08-15T16:48:51,2025-08-16T13:42:51,ORD-005,ORD-005,Order & shipping,Order & shipping,False,1


## Sanity checks

- Every ticket should have a triage prediction.
- Only closed/solved tickets should have a final prediction.
- The disagreement rate should be non-zero (drift/ambiguous cases exist by
  design) but well below 100%.

In [4]:
assert classified_df["predicted_category_triage"].notna().all(), "every ticket needs a triage prediction"

closed_mask = classified_df["status"].isin(["solved", "closed"])
assert (classified_df.loc[closed_mask, "predicted_category_final"].notna()).all(), "closed tickets need a final prediction"
assert (classified_df.loc[~closed_mask, "predicted_category_final"].isna()).all(), "open/pending tickets shouldn't have a final prediction yet"

closed = classified_df[classified_df["predicted_category_final"].notna()]
disagreement_rate = closed["category_disagreement"].mean()
print(f"{closed_mask.sum()} closed tickets have a final prediction")
print(f"Disagreement rate (triage vs final, closed tickets): {disagreement_rate:.1%}")
assert 0 < disagreement_rate < 0.5, "disagreement rate looks implausible"

958 closed tickets have a final prediction
Disagreement rate (triage vs final, closed tickets): 10.8%


In [5]:
classified_df[classified_df["category_disagreement"]][
    ["id", "subject", "predicted_category_triage", "predicted_category_final"]
].head(10)

,id,subject,predicted_category_triage,predicted_category_final
3,10003,Package hasn't arrived yet,ORD-002,GEN-003
14,10014,Delivery delay on #821024,ORD-002,RET-004
23,10023,Still waiting on refund for #385470,RET-002,RET-004
35,10035,Order #763829 missing an item,ORD-004,GEN-002
61,10061,Stock availability question,PRD-004,GEN-OTHER
68,10068,Invoice request for #160423,PAY-003,PAY-001
78,10078,Refund status,RET-002,LOY-001
82,10082,Package hasn't arrived yet,ORD-002,ORD-001
96,10096,Wrong item received,PRD-002,PRD-003
109,10109,Defective phone case,PRD-002,PRD-001


## Save for the dashboard

In [6]:
classified_df.to_csv("../outputs/tickets_classified.csv", index=False)
print("Saved outputs/tickets_classified.csv")

Saved outputs/tickets_classified.csv
